## 3. Exercise: Observability with Langfuse

In this exercise you will:
- Register a new **styled_visualizer** prompt in Langfuse Prompt Management.
- Configure the existing visualizer to use this new prompt (via `settings.env`).
- Run the existing multi-agent graph with Langfuse tracing so you can inspect prompts and traces in the Langfuse UI.

We will build on the concepts from Exercises 1 and 2 (graph execution and the visualizer agent).

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

# Construct src-path and append it to sys.path
src_path = Path.cwd().parent.parent
sys.path.append(str(src_path))

# Initialize settings-object that loads the OpenAI API, Tavily API keys and Langfuse credentials to the environment
from core.settings import settings  # noqa: F401


### 3.1 Register a `styled_visualizer` prompt in Langfuse

In this step, you will create a new prompt in **Langfuse Prompt Management** called `styled_visualizer`.
This prompt will be used by the visualizer agent and should:
- Instruct the agent to always use a **specific visual style** (colors, background, line/bar styles, etc.).
- Preserve the existing behavior of calling `run_matplotlib` and outputting `FILE_PATH` and `CHART_NOTES` lines.
- Be registered as a **chat prompt** with two roles: `system` and `user_plan`.

The existing visualizer prompt is registered in `src/prompt_management/register_prompt.ipynb` as `visualizer_prompt`.
Use that notebook as a reference for the structure (types, roles, and how to call `create_prompt`).

**Hints:**
- System prompt ideas:
  - Explain that the agent is a *Styled Chart Generator* in a multi-agent system.
  - Require use of the `run_matplotlib` tool and describe how to use `plt` and `ax`.
  - Add detailed visual style instructions, for example:
    - Use a white background with light gray grid lines.
    - Use a fixed color palette (e.g. blue, orange, green) for series.
    - Rules for line charts vs bar charts (opacity, edges, markers, etc.).
    - How to style axes and labels.
  - At the end, require the two lines `FILE_PATH: ...` and `CHART_NOTES: ...`.
- User prompt template:
  - You can follow the existing visualizer user prompt and include placeholders like `{user_query}` and `{context}`.
- For the final `create_prompt` call, mirror the structure of `visualizer_prompt` in
  `src/prompt_management/register_prompt.ipynb` (name, type, labels, and chat roles).


In [ ]:
import langfuse

from models.prompts import PromptRole

# Initialize Langfuse client (credentials come from settings imported above)
langfuse_client = langfuse.get_client()

# TODO: Define a system prompt string for the styled visualizer.
# It should include instructions about using run_matplotlib AND detailed visual style (colors, background, line/bar styles).
styled_visualizer_system_prompt = """
# <fill in>
"""

# TODO: Define a user prompt template similar to the existing visualizer user prompt.
# It should contain placeholders {user_query} and {context}.
styled_visualizer_user_prompt = """
# <fill in>
"""

# TODO: Create the prompt in Langfuse Prompt Management using the Langfuse client.
#   - name: 'styled_visualizer'
#   - type: 'chat'
#   - prompt: list with SYSTEM and USER_PLAN roles
#   - labels: ['dev_latest']
# TIP: Look at how 'visualizer_prompt' is registered in
#   src/prompt_management/register_prompt.ipynb and mirror that structure here.


After running the cell above, open the **Langfuse UI** and verify that a prompt named `styled_visualizer` exists,
with type `chat`, label `dev_latest`, and the expected system/user prompt contents (including your visual style rules).

Once this is confirmed, continue with the next section.

### 3.2 Point the existing visualizer to `styled_visualizer` via `settings.env`

The existing Visualizer agent and `VisualizerNode` use prompts from the **Prompt Registry** via
`settings.prompt_registry.visualizer`.

To use your new `styled_visualizer` prompt without changing any code in the agent or node,
you will update the configuration in `settings.env`.


**Instructions:**
1. Open the `settings.env` file in the project root.
2. Locate (or add) the section for the visualizer prompt registry settings.
3. Set the following values:

```env
prompt_registry__visualizer__name=styled_visualizer
prompt_registry__visualizer__type=chat
prompt_registry__visualizer__label=dev_latest
```

4. Save the file.
5. **Restart the notebook kernel** so that `core.settings` reloads with the updated configuration.

After restarting, you can run the cell below to inspect the effective visualizer prompt configuration.
It should now point to `styled_visualizer`.


In [ ]:
# Optional: Check that the visualizer prompt registry configuration
# points to your new 'styled_visualizer' prompt.
from core.settings import settings

settings.prompt_registry.visualizer


### 3.3 Run the graph with a Langfuse trace

Finally, you will run the existing multi-agent graph, but wrapped in a **Langfuse trace**.
Because you updated `settings.env` in the previous step, the existing visualizer agent/node will now use
the `styled_visualizer` prompt when it is invoked in the graph.

We want to:
- Create an initial `State` with a user query that exercises the visualizer.
- Run `graph.run(start_node=PlannerNode(), state=state)` inside `start_as_current_observation` and `propagate_attributes`.
- Inspect the final answer and the Langfuse UI to see the trace and prompts.


**Hints:**
- Look at `src/main.py` for an example of how the CLI wraps the graph run in a Langfuse trace using
  `get_client`, `start_as_current_observation`, and `propagate_attributes`.
- Follow the same high-level pattern in this notebook, but adapt it to use `styled_state` and `PlannerNode`.
- Use a user query that clearly requires a chart (so the visualizer is triggered).
- Enable at least `WEB_RESEARCHER`, `SYNTHESIZER`, `VISUALIZER`, and `CHART_SUMMARIZER` in `enabled_agents`.


In [ ]:
from models.agents import Agents
from models.state import Message, State

# TODO: Define a user query that requires a chart and will exercise the visualizer.
styled_user_query = (
    "Plot the current market capitalization of the top bank in Germany as a nicely styled bar chart. "
    "State the day to which the final figure refers."
)

# TODO: Choose which agents to enable. Make sure VISUALIZER is included.
styled_enabled_agents = [
    Agents.WEB_RESEARCHER,
    Agents.SYNTHESIZER,
    Agents.VISUALIZER,
    Agents.CHART_SUMMARIZER,
]

# TODO: Initialize the State for the multi-agent graph.
styled_state = State(
    messages=[Message(content=styled_user_query)],
    user_query=styled_user_query,
    enabled_agents=styled_enabled_agents,
)


In [ ]:
from uuid import uuid4

from langfuse import get_client, propagate_attributes

from agents import PlannerNode
from graph import graph

# Initialize a Langfuse client for this notebook run.
traced_langfuse_client = get_client()

# TODO: Implement the Langfuse-wrapped graph run here.
# Use src/main.py as a reference for how to:
#   - create a session id (e.g. using uuid4)
#   - call start_as_current_observation
#   - call propagate_attributes
#   - run graph.run(start_node=PlannerNode(), state=styled_state)


### 3.4 Analyse the Langfuse trace and cost

After you have implemented and run the Langfuse-wrapped graph execution, open the **Langfuse UI** and:

1. **Find the correct session / trace**:
   - Filter by the session id you used (e.g. `notebook-...`) or by the span name you chose (e.g. `styled-visualizer-notebook-run`).
   - Verify that the trace contains the full multi-agent run.
2. **Inspect the total cost of the run** and relate it to the prompts and tokens used.
3. **Check the cost per agent**:
   - Identify which agents (Planner, WebResearcher, Visualizer, Synthesizer, ChartSummarizer, Executor) were called.
   - Compare their individual token usage and cost. Are they equally expensive? Which agent dominates the cost?
4. **Follow along the agent's reasoning**:
   - Step through the chain of observations/spans in the trace.
   - For each agent invocation, read the input and output messages.
   - Verify that the visualizer used your `styled_visualizer` prompt (check the prompt text) and that its output followed the styling instructions from that prompt (colors, background, chart types, etc.).

Summarise your findings for yourself: how expensive is a full run, which parts cost the most, and does the reasoning flow make sense end-to-end?
